# Chat Format and SFT


Use this notebook after the chat format and SFT note. The goal is to show that chat fine-tuning is still causal language modeling, but over a role-formatted conversation string.

You will:
- turn messages into one token stream
- compare a base checkpoint against a small SFT run
- verify that the architecture stays the same while the data distribution changes


In [1]:
import sys
from pathlib import Path

ROOT = next((path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'course_tools').exists() or (path / 'picollm').exists()), Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import copy
from course_tools import DEFAULT_CHAT_MESSAGES, build_demo_bundle, evaluate_model, format_messages, train_model

LECTURE_NOTE_TITLE = 'Chat Format and SFT'
print(f'Lecture note: {LECTURE_NOTE_TITLE}')


Lecture note: Chat Format and SFT


## A conversation becomes one causal token stream


In [2]:
formatted = format_messages(DEFAULT_CHAT_MESSAGES + [{'role': 'assistant', 'content': 'Self-attention lets tokens mix information from relevant context.'}])
print(formatted)


<|system|>
You are a compact teaching assistant for an LLM course.
<|user|>
Explain self-attention in three short sentences.
<|assistant|>
Self-attention lets tokens mix information from relevant context.



## Base model versus SFT model


In [3]:
bundle = build_demo_bundle(steps=20)
tokenizer = bundle['tokenizer']
target_answer = 'Self-attention creates a weighted summary of earlier tokens.'
prompt = format_messages(DEFAULT_CHAT_MESSAGES, add_assistant_prompt=True)
target_chat = format_messages(
    DEFAULT_CHAT_MESSAGES + [{'role': 'assistant', 'content': target_answer}]
)
base_metrics = evaluate_model(bundle['model'], tokenizer, text='\n'.join([target_chat] * 8), batch_size=4)
print('base eval on chat-style target:', base_metrics)

sft_model = copy.deepcopy(bundle['model'])
sft_text = '\n'.join([target_chat] * 24)
history = train_model(sft_model, tokenizer, train_text=sft_text, eval_text=sft_text, steps=30, learning_rate=2e-3, batch_size=4)
print('sft history:', history)
sft_metrics = evaluate_model(sft_model, tokenizer, text='\n'.join([target_chat] * 8), batch_size=4)
print('sft eval on same target:', sft_metrics)


base eval on chat-style target: {'loss': 6.245710849761963, 'bpb': 9.010656069777978}
sft history: [{'step': 1.0, 'train_loss': 5.743241310119629, 'eval_loss': 5.198031902313232, 'bpb': 7.499174847849925}, {'step': 6.0, 'train_loss': 5.3510236740112305, 'eval_loss': 4.945338726043701, 'bpb': 7.134615655579392}, {'step': 12.0, 'train_loss': 4.138543605804443, 'eval_loss': 3.879585027694702, 'bpb': 5.597058080162219}, {'step': 18.0, 'train_loss': 3.5885887145996094, 'eval_loss': 3.314464807510376, 'bpb': 4.781761940996212}, {'step': 24.0, 'train_loss': 3.1823978424072266, 'eval_loss': 3.034261703491211, 'bpb': 4.377514312386069}, {'step': 30.0, 'train_loss': 2.765615940093994, 'eval_loss': 2.847759485244751, 'bpb': 4.108448487007109}]
sft eval on same target: {'loss': 2.878204584121704, 'bpb': 4.152371480176264}


## The architecture stayed the same and the training distribution changed


In [4]:
print(type(bundle['model']).__name__)
print(type(sft_model).__name__)
print('same architecture, different parameters after extra chat-style training')


TinyTransformerLM
TinyTransformerLM
same architecture, different parameters after extra chat-style training


## External reference repos


**RASBT**
- https://github.com/rasbt/LLMs-from-scratch/blob/main/ch07/01_main-chapter-code/gpt_instruction_finetuning.py
**NANOCHAT**
- https://github.com/karpathy/nanochat/blob/master/scripts/chat_sft.py
- https://github.com/karpathy/nanochat/blob/master/nanochat/tokenizer.py
